# 03 - STF gallery

A **source time function** (STF) describes how slip is released in time at a
source. ShakerMaker convolves it with the Green's function after the FK
computation. This notebook builds and plots the five STFs shipped with
ShakerMaker:

- **Dirac** - an impulse, used to get the raw Green's function.
- **Discrete** - any user-supplied (t, value) curve.
- **Brune** - the classic omega-square far-field pulse (original + smoothed).
- **Gaussian** - a smooth bell (here with the LOH.1 parameters).
- **SRF2** - a slip-rate function with rise, plateau and decay phases.

Each STF generates its data once you set its `.dt`; afterwards `.t` and
`.data` are numpy arrays. Every figure is saved as a `.png` in this folder.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from shakermaker.stf_extensions.dirac import Dirac
from shakermaker.stf_extensions.discrete import Discrete
from shakermaker.stf_extensions.brune import Brune
from shakermaker.stf_extensions.gaussian import Gaussian
from shakermaker.stf_extensions.srf2 import SRF2

plt.style.use("ggplot")
DT = 0.001

## Dirac

A single unit sample at `t = 0`. Drawn as a stem since it is an impulse.

In [ ]:
stf = Dirac()
stf.dt = DT

fig, ax = plt.subplots(figsize=(6, 2.5))
ax.stem(stf.t, stf.data, linefmt="#E74C3C", markerfmt="o", basefmt=" ")
ax.set_title("Dirac")
ax.set_xlabel("Time, t (s)")
ax.set_ylabel("STF")
fig.tight_layout()
fig.savefig("stf_dirac.png", dpi=150, bbox_inches="tight")

## Discrete

Any curve you supply. Here a narrow Gaussian-shaped pulse sampled on a
user grid; it is interpolated to the simulation `dt` internally.

In [ ]:
t_user = np.linspace(0, 0.3, 301)
data_user = np.exp(-((t_user - 0.1) / 0.02) ** 2)
stf = Discrete(data_user, t_user)
stf.dt = DT

fig, ax = plt.subplots(figsize=(6, 2.5))
ax.plot(stf.t, stf.data, color="#E74C3C", linewidth=1.5)
ax.set_title("Discrete")
ax.set_xlabel("Time, t (s)")
ax.set_ylabel("STF")
fig.tight_layout()
fig.savefig("stf_discrete.png", dpi=150, bbox_inches="tight")

## Brune (original + smoothed)

The omega-square far-field source pulse. The `smoothed=True` variant removes
the sharp onset.

In [ ]:
f0, t0 = 10.0, 0.5
stf = Brune(f0=f0, t0=t0, slip=1.0, smoothed=False)
stf.dt = DT
stf_s = Brune(f0=f0, t0=t0, slip=1.0, smoothed=True)
stf_s.dt = DT

fig, ax = plt.subplots(figsize=(6, 2.5))
ax.plot(stf.t, stf.data, color="#E74C3C", linewidth=1.5, label="Original")
ax.plot(stf_s.t, stf_s.data, color="#3498DB", linewidth=1.5, label="Smoothed")
ax.set_title(f"Brune. f0={f0} Hz, t0={t0} s")
ax.set_xlabel("Time, t (s)")
ax.set_ylabel("STF")
ax.legend()
fig.tight_layout()
fig.savefig("stf_brune.png", dpi=150, bbox_inches="tight")

## Gaussian (LOH.1 parameters)

Built with the LOH.1 convention: width `sigma = 0.06 s`, time shift
`t0 = 6*sigma`, and frequency parameter `freq = 1/sigma`.

In [ ]:
sigma = 0.06
stf = Gaussian(t0=6 * sigma, freq=1 / sigma, M0=1.0, derivative=False)
stf.dt = DT

fig, ax = plt.subplots(figsize=(6, 2.5))
ax.plot(stf.t, stf.data, color="#E74C3C", linewidth=1.5)
ax.set_title(f"Gaussian (LOH.1). sigma={sigma} s")
ax.set_xlabel("Time, t (s)")
ax.set_ylabel("STF")
fig.tight_layout()
fig.savefig("stf_gaussian.png", dpi=150, bbox_inches="tight")

## SRF2

A slip-rate function with a rising sine branch, a `sqrt(a + b/t^2)` plateau,
and a decaying sine tail; normalized to unit area then scaled by `slip`.

In [ ]:
stf = SRF2(Tr=2.0, Tp=0.1, Te=1.5, dt=DT, slip=12.0, a=1.0, b=1.0)
stf.dt = DT

fig, ax = plt.subplots(figsize=(6, 2.5))
ax.plot(stf.t, stf.data, color="#E74C3C", linewidth=1.5)
ax.set_title("SRF2")
ax.set_xlabel("Time, t (s)")
ax.set_ylabel("STF")
fig.tight_layout()
fig.savefig("stf_srf2.png", dpi=150, bbox_inches="tight")

## All five together

A small grid overview of every STF on its own axes.

In [ ]:
dirac = Dirac(); dirac.dt = DT
disc = Discrete(data_user, t_user); disc.dt = DT
brune = Brune(f0=10.0, t0=0.5, slip=1.0, smoothed=False); brune.dt = DT
gauss = Gaussian(t0=6 * sigma, freq=1 / sigma, M0=1.0); gauss.dt = DT
srf2 = SRF2(Tr=2.0, Tp=0.1, Te=1.5, dt=DT, slip=12.0, a=1.0, b=1.0); srf2.dt = DT

gallery = [("Dirac", dirac), ("Discrete", disc), ("Brune", brune),
           ("Gaussian", gauss), ("SRF2", srf2)]

fig, axes = plt.subplots(2, 3, figsize=(12, 5))
for ax, (name, s) in zip(axes.ravel(), gallery):
    if name == "Dirac":
        ax.stem(s.t, s.data, linefmt="#E74C3C", markerfmt="o", basefmt=" ")
    else:
        ax.plot(s.t, s.data, color="#E74C3C", linewidth=1.5)
    ax.set_title(name)
    ax.set_xlabel("Time, t (s)")
    ax.set_ylabel("STF")
axes.ravel()[-1].axis("off")
fig.tight_layout()
fig.savefig("stf_gallery.png", dpi=150, bbox_inches="tight")